# Mammogram Chatbot with RAG
This notebook implements a Retrieval-Augmented Generation (RAG) chatbot that processes PDFs and Images to answer questions based on mammogram reports.

## 1. Prerequisites
Run the cell below to install all required libraries. You may also need to install `poppler` on your system for PDF processing.

In [ ]:
!pip install -r requirements.txt

## 2. Imports and Configuration
Setting up the environment and local paths.

In [ ]:
import gradio as gr
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import ctransformers
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
from transformers import AutoTokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
import easyocr
import torch
import os
import numpy as np
from PIL import Image
import shutil

# Configuration
DATA_PATH = './data/'
TEMP_FOLDER = os.path.join(DATA_PATH, "TEMP")
Img_folder = os.path.join(TEMP_FOLDER, "images")
VECTORSTORE_DIR = './vectorstore'
PDF_DB_PATH = os.path.join(VECTORSTORE_DIR, 'db_pdf')
IMAGE_DB_PATH = os.path.join(VECTORSTORE_DIR, 'db_image')
MERGED_DB_PATH = os.path.join(VECTORSTORE_DIR, 'merged_db')

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(Img_folder, exist_ok=True)
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 3. Load the Language Model
We load the Mistral model once to ensure efficiency.

In [ ]:
def load_llm():
    model_path = "./mistral-7b-instruct-v0.2.Q5_K_M.gguf"
    if not os.path.exists(model_path):
        print(f"Warning: Model file not found at {model_path}")
    
    return ctransformers.CTransformers(
        model=model_path,
        model_type="mistral",
        max_new_tokens=512,
        temperature=0.5
    )

global_llm = load_llm()
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': device})
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", clean_up_tokenization_spaces=True)
reader = easyocr.Reader(['en'])

## 4. Processing Logic
Functions for PDF vectorization, OCR, and vector store merging.

In [ ]:
def create_vector_db(pdf_path):
    try:
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        output_folder = os.path.join(TEMP_FOLDER, pdf_name)
        os.makedirs(output_folder, exist_ok=True)
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
        texts = text_splitter.split_documents(documents)
        db = FAISS.from_documents(texts, embeddings)
        db.save_local(PDF_DB_PATH)
        images = convert_from_path(pdf_path)
        image_paths = []
        for i, img in enumerate(images):
            img_path = os.path.join(output_folder, f'page_{i + 1}.png')
            img.save(img_path, 'PNG')
            image_paths.append(img_path)
        merge_message = merge_vector_stores()
        return f"Successfully vectorized {len(images)} pages. {merge_message}", image_paths
    except Exception as e:
        return f"Error processing PDF: {e}", []

def extract_text_from_image(image):
    image_array = np.array(image)
    result = reader.readtext(image_array)
    return " ".join([text for _, text, _ in result])

def vectorize_image_text(image_text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20) 
    text_chunks = splitter.split_text(image_text) 
    db = FAISS.from_texts(text_chunks, embeddings)
    db.save_local(IMAGE_DB_PATH)
    return merge_vector_stores()

def merge_vector_stores():
    try:
        pdf_exists = os.path.exists(PDF_DB_PATH)
        image_exists = os.path.exists(IMAGE_DB_PATH)
        if not pdf_exists and not image_exists:
            return "No vector store exists to merge."
        db_pdf = FAISS.load_local(PDF_DB_PATH, embeddings, allow_dangerous_deserialization=True) if pdf_exists else None
        db_image = FAISS.load_local(IMAGE_DB_PATH, embeddings, allow_dangerous_deserialization=True) if image_exists else None
        if db_pdf and db_image:
            db_pdf.merge_from(db_image)
            db_pdf.save_local(MERGED_DB_PATH)
            return "Merged vector stores successfully."
        elif db_pdf:
            db_pdf.save_local(MERGED_DB_PATH)
            return "Saved PDF store to merged database."
        elif db_image:
            db_image.save_local(MERGED_DB_PATH)
            return "Saved Image store to merged database."
        return "No data to merge."
    except Exception as e:
        return f"Error merging vector stores: {e}"

def final_result(query):
    try:
        if not os.path.exists(MERGED_DB_PATH):
            return "Please upload a document first."
        db = FAISS.load_local(MERGED_DB_PATH, embeddings, allow_dangerous_deserialization=True)
        qa = RetrievalQA.from_chain_type(
            llm=global_llm,
            chain_type='stuff',
            retriever=db.as_retriever(search_kwargs={'k': 1}),
            return_source_documents=True
        )
        response = qa({'query': query})
        return response['result']
    except Exception as e:
        return f"Error: {e}" 

## 5. Gradio Interface
Launching the web interface.

In [ ]:
def handle_file_upload(file_paths, from_explorer=False):
    status_messages = []
    all_images = []
    if not file_paths: return None, "No files selected.", [], gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
    for file_path in file_paths:
        if file_path.lower().endswith('.pdf'):
            status, images = create_vector_db(file_path)
            status_messages.append(status)
            all_images.extend(images)
        else:
            try:
                image = Image.open(file_path)
                all_images.append(image)
                extracted_text = extract_text_from_image(image)
                merge_message = vectorize_image_text(extracted_text)
                status_messages.append(f"Image processed: {merge_message}")
            except Exception as e:
                status_messages.append(f"Error processing {file_path}: {e}")
    return None, "\n".join(status_messages), all_images, gr.update(visible=True), gr.update(visible=True), gr.update(visible=True)

with gr.Blocks() as app:
    chat_history = gr.State([])
    with gr.Row():
        with gr.Column(scale=1):
            file_explorer_input = gr.FileExplorer(label="Select from Folder", root_dir=DATA_PATH)
            file_input = gr.File(label="Upload Files", file_count="multiple")
        with gr.Column(scale=1):
            image_preview = gr.Image(label="Preview")
            gallery_output = gr.Gallery(label="Pages", visible=False)
            status = gr.Textbox(label="Status")
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat")
            msg = gr.Textbox(label="Question", visible=False)
            submit = gr.Button("Submit", visible=False)

    def respond(q, h):
        a = final_result(q)
        h.append((q, a))
        return h, h, ""

    file_input.change(lambda f: handle_file_upload([x.name for x in f] if f else []), inputs=file_input, outputs=[image_preview, status, gallery_output, gallery_output, msg, submit])
    submit.click(respond, [msg, chat_history], [chatbot, chat_history, msg])

app.launch()